In [ ]:
import os
import json
import gradio as gr
from win11toast import toast
from dotenv import load_dotenv
from abc import ABC, abstractmethod
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import display, HTML


In [ ]:
display(HTML("""
<table width='100%'>
    <tr>
        <td width='33%' style='text-align:center;'><b><font style='color:lightgreen'>Tool Calling Pattern</font><b></td>
        <td width='33%' style='text-align:center;'><b><font style='color:lightgreen'>Tool Calling Theory (Perception)</font><b></td>
        <td width='33%' style='text-align:center;'><b><font style='color:lightgreen'>Tool Calling Actual (Execution)</font><b></td>
    </tr>
    <tr>
        <td width='30%' style='align:center;'><img src='../../images/agentic-tool-calling.png' style='width:98%;height:150px;'/></td>
        <td width='30%' style='align:center;'><img src='../../images/agentic-tool-calling-theory.png' style='width:98%;height:150px;'/></td>
        <td width='30%' style='align:center;'><img src='../../images/agentic-tool-calling-practice.png' style='width:98%;height:150px;'/></td>
    </tr>
</table>
"""))

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai_api_key_found = openai_api_key and openai_api_key.startswith('sk-proj-') and len(openai_api_key) > len('sk-proj-')

if openai_api_key_found:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please setup OPENAI_API_KEY in .env file.")
    print(f"""Generate your free trial OpenAI Api key here: https://platform.openai.com/api-keys""")
    raise Exception("Please set Open API Key to proceed.")

In [ ]:
PRINT_ENABLED = False

def printHtmlContentWithHeader(header, content, displayFlag = True):
    printLog = displayFlag
    if PRINT_ENABLED:
        printLog = PRINT_ENABLED and displayFlag
    
    if printLog:    
        display(HTML(f"""
        <h2>{header}</h2>
        <p>{content}</p>
        </br>
        """))

# Read me/User-Profile.pdf and extract user information from the document.
def readProfile():
    text = ""
    reader = PdfReader("../../me/user-profile.pdf")
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t

    if len(text) > 0:
        # Clean up data
        text = text.strip().replace("\n", " ").replace("• • • • • • • • • • • • • • • \u200b \u200b \u200b\u200b \u200b \u200b", "")
    printHtmlContentWithHeader("User Profile Profile", text, False)
    return text

# Short summary about the user read from me/summary.txt
def readSummary():
    summaryText = ""
    with open("../../me/summary.txt", "r", encoding="utf-8") as f:
        summaryText = f.read()
    if len(summaryText) > 0:
        summaryText = summaryText.strip().replace("\n", " ")
    printHtmlContentWithHeader("Summary", summaryText, False)
    return summaryText

# Populate User Context data
USER_NAME = "ETHAN HUNT"
PROFILE_DATA = readProfile()
SUMMARY = readSummary()


In [ ]:
class GuardEvaluationResponse:
    is_acceptable: bool
    feedback: str

    def __init__(self):
        self.is_acceptable = True
        self.feedback = ""
    
class GuardEvaluatorBase(ABC):
    @abstractmethod
    def evaluate(self, msg) -> GuardEvaluationResponse:
        pass


In [ ]:

class ToolsHandler():

    # Sends toast notification on Win11 machine. If you have MAC or Linux please updaate this method to handle specific sytem. 
    def sendNotification(self, title, msg):
        try:
            toast(title,  msg)
        except:
            print('Ignoring: Win11 Toast Library Error')


    # This Tool (1): could log message to diary or store of specific choice her we just choose to send Toast Notification.
    def record_unknown_question(self, question):
        self.sendNotification(title="User Query On Un-Known Topic", msg=f"Recording {question} asked that I couldn't answer")
        return {"recorded": "ok"}

    # This Tool (2): could log message to diary or store of specific choice her we just choose to send Toast Notification.
    def record_user_details(self, email, name="Name not provided", notes="not provided"):
        self.sendNotification(title=f"User connect request", msg= f"Recording interest from {name} with email {email} and notes {notes}")
        return {"recorded": "ok"}

    # This function can take a list of tool calls, and run them. This is the IF statement!!
    def handle_tool_calls(self, tool_calls):
        results = []
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            print(f"Tool called: {tool_name}", flush=True)

            # THE BIG IF STATEMENT!!!

            if tool_name == "record_user_details":
                result = self.record_user_details(**arguments)
            elif tool_name == "record_unknown_question":
                result = self.record_unknown_question(**arguments)

            results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
        return results

    def getToolsMeta(self) -> []:
        record_user_details_json = {
        "name": "record_user_details",
        "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {
                    "type": "string",
                    "description": "The email address of this user"
                },
                "name": {
                    "type": "string",
                    "description": "The user's name, if they provided it"
                }
                ,
                "notes": {
                    "type": "string",
                    "description": "Any additional information about the conversation that's worth recording to give context"
                }
            },
            "required": ["email"],
            "additionalProperties": False
        }}

        record_unknown_question_json = {
        "name": "record_unknown_question",
        "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question that couldn't be answered"
                },
            },
            "required": ["question"],
            "additionalProperties": False
        }}

        tools = [
            {"type": "function", "function": record_user_details_json},
            {"type": "function", "function": record_unknown_question_json}
        ]

        return tools


In [ ]:

class UserProfileAgent:
    # Initalize OPEN AI
    openai= OpenAI()
 
    tool_handler: ToolsHandler

    # Constructor method
    def __init__(self, 
    usrName: str, 
    profileData: str, 
    summaryText: str,
    toolHandler: ToolsHandler
    ):
    
        self.user_name = usrName
        self.user_profile = profileData
        self.user_summary = summaryText
        self.tool_handler = toolHandler
        
        # ************************* System Prompt specifies the tool to use if answer is not knowm ********************** #
        # *** Although its not nessecary to define as it is already defined in tools metadata and LLM is capable to discover that its good to be specific *** #  
        self.system_prompt = f"You are acting as {usrName}. You are answering questions on {usrName}'s website, \
        particularly questions related to {usrName}'s career, background, skills and experience. \
        Your responsibility is to represent {usrName} for interactions on the website as faithfully as possible. \
        You are given a summary of {usrName}'s background and LinkedIn profile which you can use to answer questions. \
        Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
        If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
        If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

        self.system_prompt += f"\n\n## Summary:\n{summaryText}\n\n## LinkedIn Profile:\n{profileData}\n\n"
        self.system_prompt += f"With this context, please chat with the user, always staying in character as {usrName}."
        
    
    # Use OPENAI and gpt-4o-mini to compute response to user query #
        
    def askQuery(self, message, rep, fdbk, hist) -> str:
        
        messages = [{"role": "system", "content": self.system_prompt}] + [{"role": "user", "content": message}]
        done = False
        while not done:

            # This is the call to the LLM - see that we pass in the tools json
            response = self.openai.chat.completions.create(
                model="gpt-4o-mini", 
                messages=messages, 
                tools=self.tool_handler.getToolsMeta()) # Pass the tools metadata to LLM 

            finish_reason = response.choices[0].finish_reason
            
            # If the LLM wants to call a tool, we do that!. LLM will not directly execute the tool It will ask tool call 
            # to be handled on its behalf and expects back a response if tool(s) was/were executed, with out put of tool execution if any
            # Finally LLM responds to user if it has not tools to compute the outcome.
            if finish_reason=="tool_calls":
                message = response.choices[0].message
                tool_calls = message.tool_calls
                # Delegate identified tool execution. LLM never executes tool directly. So effectively we end up writing 
                # Code for handling itentified tool execution Which essentially a large block of if statements.
                print(f"LLM Identifed tool(s) for Execution: {tool_calls}")
                results = self.tool_handler.handle_tool_calls(tool_calls=tool_calls)
                messages.append(message)
                messages.extend(results)
            else:
                done = True
                
        return response.choices[0].message.content

In [ ]:
userProfileAgent = UserProfileAgent(USER_NAME, 
PROFILE_DATA, 
SUMMARY, 
ToolsHandler())

gr.ChatInterface(fn = userProfileAgent.askQuery).launch()